# Phase 3.2 — Economic Realism Stress Test

**Goal**: identify whether profit-optimal cutoffs remain approve-all (Phase 3.1B
finding) or become interior under realistic cost / APR / LGD / PD stress.

**Stress dimensions** (locked by user prompt):
- PD multipliers: {1, 2, 3, 5}
- cost_of_funds_annual: {0.00, 0.03, 0.06}
- acquisition_cost (per approved loan): {0, 250, 500}
- LGD: {0.55, 0.65, 0.75, 0.85}
- APR strategy: {tiered_uncapped, tiered_cap_24, flat_18, flat_24}

Combined grid = 4 × 3 × 3 × 4 × 4 = **576 cells** (primary PD only).

**Per cell**: mean / total / share>0 of Expected_Profit, profit-optimal cutoff,
Youden cutoff (computed once per PD model on OOT), gap.

**Classification**:
- A. approve-all: profit-optimal k* >= 99%
- B. interior-optimum: 50% <= k* < 99%
- C. reject-most: k* < 50%

In [1]:
import sys, time, json, joblib, warnings
from pathlib import Path

import numpy as np
import pandas as pd
warnings.filterwarnings("ignore")

REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from src.economics import (
    batch_lifetime_economics,
    apr_tier_lookup_vec, apr_tier_lookup_capped_vec,
)
from src.evaluation import compute_gini

# Load Phase 3.1B per-account economics (already filtered to 24m+36m,
# already scored with all 4 PD models)
ECO_PATH = REPO_ROOT / "artifacts/economic_framework/economics_per_account.parquet"
eco = pd.read_parquet(ECO_PATH)
print(f"Loaded eco data: {eco.shape}")
print(f"  splits: {eco['split_new'].value_counts().to_dict()}")

PRIMARY_PD = "pd_lgb_platt"
PD_COLS = ["pd_lgb_platt", "pd_lr_full_platt", "pd_lr_nof6e_platt", "pd_sc_platt"]

T0 = time.time()

Loaded eco data: (235968, 15)
  splits: {'train_for_model': 150476, 'oot': 64027, 'calib': 21465}


## 1. APR strategy helpers

In [2]:
def apr_array(pd_arr, strategy):
    """Return per-row APR array given a strategy name."""
    if strategy == "tiered_uncapped":
        return apr_tier_lookup_vec(pd_arr)
    if strategy == "tiered_cap_24":
        return apr_tier_lookup_capped_vec(pd_arr, cap=0.24)
    if strategy == "tiered_cap_18":
        return apr_tier_lookup_capped_vec(pd_arr, cap=0.18)
    if strategy.startswith("flat_"):
        v = int(strategy.split("_")[1]) / 100.0
        return np.full_like(pd_arr, v, dtype=np.float64)
    raise ValueError(f"unknown strategy {strategy}")

# Sanity
test = np.array([0.001, 0.01, 0.04, 0.10])
for s in ["tiered_uncapped","tiered_cap_24","tiered_cap_18","flat_18","flat_24"]:
    print(f"  {s:<18}: {apr_array(test, s)}")

  tiered_uncapped   : [0.12 0.22 0.26 0.3 ]
  tiered_cap_24     : [0.12 0.22 0.24 0.24]
  tiered_cap_18     : [0.12 0.18 0.18 0.18]
  flat_18           : [0.18 0.18 0.18 0.18]
  flat_24           : [0.24 0.24 0.24 0.24]


## 2. Youden cutoffs (per PD model, computed once on OOT)

In [3]:
from sklearn.metrics import roc_curve

oot = eco[eco["split_new"] == "oot"].copy()
print(f"OOT rows: {len(oot):,}")
youden_per_pd = {}
for pd_col in PD_COLS:
    if oot[pd_col].notna().all():
        fpr, tpr, thr = roc_curve(oot["default_flag_12m"], oot[pd_col])
        j = tpr - fpr
        thr_star = float(thr[np.argmax(j)])
        oot_approve_pct = float((oot[pd_col] <= thr_star).mean() * 100)
        # Apply same threshold on full eco pop
        full_approve_pct = float((eco[pd_col].fillna(1.0) <= thr_star).mean() * 100)
        youden_per_pd[pd_col] = {
            "threshold": thr_star,
            "oot_approve_pct": oot_approve_pct,
            "full_approve_pct": full_approve_pct,
        }
        print(f"  {pd_col:<22}  Youden_thr={thr_star:.4f}  OOT_acc={oot_approve_pct:.2f}%  full_acc={full_approve_pct:.2f}%")

OOT rows: 64,027
  pd_lgb_platt            Youden_thr=0.0106  OOT_acc=79.59%  full_acc=79.41%
  pd_lr_full_platt        Youden_thr=0.0213  OOT_acc=77.04%  full_acc=76.72%
  pd_lr_nof6e_platt       Youden_thr=0.0226  OOT_acc=80.62%  full_acc=80.59%
  pd_sc_platt             Youden_thr=0.0185  OOT_acc=87.92%  full_acc=87.90%


## 3. Combined stress grid (576 cells)

In [4]:
PD_MULT_GRID  = [1.0, 2.0, 3.0, 5.0]
COF_GRID      = [0.00, 0.03, 0.06]
ACQ_GRID      = [0, 250, 500]
LGD_GRID      = [0.55, 0.65, 0.75, 0.85]
APR_STRATS    = ["tiered_uncapped", "tiered_cap_24", "flat_18", "flat_24"]
DISC_BASE     = 0.0
OP_COST_BASE  = 0.0  # holding op_cost separate; the user's spec says discount/op grids
                     # were Phase 3.1B; this stage focuses on COF/acq/PD-mult/LGD/APR

print(f"Grid size: {len(PD_MULT_GRID)} × {len(COF_GRID)} × {len(ACQ_GRID)} × {len(LGD_GRID)} × {len(APR_STRATS)} = {len(PD_MULT_GRID)*len(COF_GRID)*len(ACQ_GRID)*len(LGD_GRID)*len(APR_STRATS)}")

L = eco["loan_amount"].to_numpy()
n = eco["n_installments"].to_numpy()
pd_base = eco[PRIMARY_PD].fillna(0.0).to_numpy()

results = []
t = time.time()
for mult in PD_MULT_GRID:
    pd_stressed = np.clip(pd_base * mult, 0.0, 0.999)
    for strat in APR_STRATS:
        apr_arr = apr_array(pd_stressed, strat)  # APR strategy uses STRESSED PD for tier
        for cof in COF_GRID:
            for acq in ACQ_GRID:
                for lgd in LGD_GRID:
                    out = batch_lifetime_economics(
                        pd_12m=pd_stressed,
                        loan_amount=L,
                        n_installments=n,
                        apr=apr_arr,
                        lgd=lgd,
                        op_cost_annual=OP_COST_BASE,
                        discount_annual=DISC_BASE,
                        cost_of_funds_annual=cof,
                        acquisition_cost=float(acq),
                    )
                    profit = out["Expected_Profit"]
                    # Cut-off: sort by stressed PD ascending, find max cumulative profit
                    order = np.argsort(pd_stressed)
                    profit_sorted = profit[order]
                    cum_profit = np.cumsum(profit_sorted)
                    if cum_profit.max() <= 0:
                        # No positive cumulative profit anywhere -> reject all
                        k_star = 0
                        cutoff_pd_star = 0.0
                        max_profit = 0.0
                    else:
                        k_star = int(np.argmax(cum_profit)) + 1
                        cutoff_pd_star = float(pd_stressed[order[k_star - 1]])
                        max_profit = float(cum_profit[k_star - 1])
                    approve_pct_star = 100.0 * k_star / len(profit)

                    youden_full_pct = youden_per_pd[PRIMARY_PD]["full_approve_pct"]
                    cutoff_gap = approve_pct_star - youden_full_pct

                    if approve_pct_star >= 99.0:
                        category = "approve_all"
                    elif approve_pct_star >= 50.0:
                        category = "interior"
                    else:
                        category = "reject_most"

                    results.append({
                        "pd_multiplier": mult,
                        "apr_strategy": strat,
                        "cost_of_funds_annual": cof,
                        "acquisition_cost": acq,
                        "lgd": lgd,
                        "mean_pd_stressed": float(pd_stressed.mean()),
                        "mean_LT_EL": float(out["LT_EL"].mean()),
                        "mean_LT_margin": float(out["LT_margin"].mean()),
                        "mean_Expected_Profit": float(profit.mean()),
                        "total_Expected_Profit": float(profit.sum()),
                        "share_profit_gt_0": float((profit > 0).mean()),
                        "k_star_approve_pct": approve_pct_star,
                        "cutoff_pd_star": cutoff_pd_star,
                        "max_total_profit_at_k_star": max_profit,
                        "youden_approve_pct_full": youden_full_pct,
                        "cutoff_gap_profit_minus_youden": cutoff_gap,
                        "category": category,
                    })
stress = pd.DataFrame(results)
print(f"\nGrid wall: {time.time()-t:.1f}s")
print(f"Total cells: {len(stress)}")

Grid size: 4 × 3 × 3 × 4 × 4 = 576



Grid wall: 135.6s
Total cells: 576


## 4. Classification distribution

In [5]:
cat_counts = stress["category"].value_counts()
print("Scenario classification:")
for c in ["approve_all", "interior", "reject_most"]:
    n = int(cat_counts.get(c, 0))
    pct = 100.0 * n / len(stress)
    print(f"  {c:<14}: {n:>4} cells ({pct:.1f}%)")

print("\nClassification x APR strategy:")
print(stress.pivot_table(index="apr_strategy", columns="category",
                          values="mean_Expected_Profit", aggfunc="count",
                          fill_value=0).to_string())

print("\nClassification x PD multiplier:")
print(stress.pivot_table(index="pd_multiplier", columns="category",
                          values="mean_Expected_Profit", aggfunc="count",
                          fill_value=0).to_string())

print("\nClassification x cost_of_funds:")
print(stress.pivot_table(index="cost_of_funds_annual", columns="category",
                          values="mean_Expected_Profit", aggfunc="count",
                          fill_value=0).to_string())

print("\nClassification x acquisition_cost:")
print(stress.pivot_table(index="acquisition_cost", columns="category",
                          values="mean_Expected_Profit", aggfunc="count",
                          fill_value=0).to_string())

print("\nClassification x lgd:")
print(stress.pivot_table(index="lgd", columns="category",
                          values="mean_Expected_Profit", aggfunc="count",
                          fill_value=0).to_string())

Scenario classification:
  approve_all   :  220 cells (38.2%)
  interior      :  356 cells (61.8%)
  reject_most   :    0 cells (0.0%)

Classification x APR strategy:
category         approve_all  interior
apr_strategy                          
flat_18                   34       110
flat_24                   55        89
tiered_cap_24             55        89
tiered_uncapped           76        68

Classification x PD multiplier:
category       approve_all  interior
pd_multiplier                       
1.0                    138         6
2.0                     72        72
3.0                     10       134
5.0                      0       144

Classification x cost_of_funds:
category              approve_all  interior
cost_of_funds_annual                       
0.00                           87       105
0.03                           74       118
0.06                           59       133

Classification x acquisition_cost:
category          approve_all  interior
acquisition_cos

## 5. Drivers — sensitivity of k_star to each parameter

In [6]:
# How much does k_star vary when each parameter moves, holding others at central values?
# Central: mult=2, COF=0.03, acq=250, LGD=0.65, APR=tiered_cap_24
def vary(param, values, fix=None):
    fix = fix or {}
    base_filter = (
        (stress["pd_multiplier"]    == fix.get("pd_multiplier",   2.0)) &
        (stress["cost_of_funds_annual"] == fix.get("cost_of_funds_annual", 0.03)) &
        (stress["acquisition_cost"] == fix.get("acquisition_cost", 250)) &
        (stress["lgd"]              == fix.get("lgd",             0.65)) &
        (stress["apr_strategy"]     == fix.get("apr_strategy",    "tiered_cap_24"))
    )
    # Replace param's filter with "any of values"
    if param == "pd_multiplier":
        slc = stress[
            (stress["cost_of_funds_annual"] == fix.get("cost_of_funds_annual", 0.03)) &
            (stress["acquisition_cost"] == fix.get("acquisition_cost", 250)) &
            (stress["lgd"] == fix.get("lgd", 0.65)) &
            (stress["apr_strategy"] == fix.get("apr_strategy", "tiered_cap_24")) &
            (stress["pd_multiplier"].isin(values))
        ]
    elif param == "cost_of_funds_annual":
        slc = stress[
            (stress["pd_multiplier"] == fix.get("pd_multiplier", 2.0)) &
            (stress["acquisition_cost"] == fix.get("acquisition_cost", 250)) &
            (stress["lgd"] == fix.get("lgd", 0.65)) &
            (stress["apr_strategy"] == fix.get("apr_strategy", "tiered_cap_24")) &
            (stress["cost_of_funds_annual"].isin(values))
        ]
    elif param == "acquisition_cost":
        slc = stress[
            (stress["pd_multiplier"] == fix.get("pd_multiplier", 2.0)) &
            (stress["cost_of_funds_annual"] == fix.get("cost_of_funds_annual", 0.03)) &
            (stress["lgd"] == fix.get("lgd", 0.65)) &
            (stress["apr_strategy"] == fix.get("apr_strategy", "tiered_cap_24")) &
            (stress["acquisition_cost"].isin(values))
        ]
    elif param == "lgd":
        slc = stress[
            (stress["pd_multiplier"] == fix.get("pd_multiplier", 2.0)) &
            (stress["cost_of_funds_annual"] == fix.get("cost_of_funds_annual", 0.03)) &
            (stress["acquisition_cost"] == fix.get("acquisition_cost", 250)) &
            (stress["apr_strategy"] == fix.get("apr_strategy", "tiered_cap_24")) &
            (stress["lgd"].isin(values))
        ]
    elif param == "apr_strategy":
        slc = stress[
            (stress["pd_multiplier"] == fix.get("pd_multiplier", 2.0)) &
            (stress["cost_of_funds_annual"] == fix.get("cost_of_funds_annual", 0.03)) &
            (stress["acquisition_cost"] == fix.get("acquisition_cost", 250)) &
            (stress["lgd"] == fix.get("lgd", 0.65)) &
            (stress["apr_strategy"].isin(values))
        ]
    return slc[[param, "k_star_approve_pct", "mean_Expected_Profit", "category"]].sort_values(param)

print("Holding other params at center (mult=2, COF=0.03, acq=250, LGD=0.65, APR=tiered_cap_24):")
print()
for p, vals in [
    ("pd_multiplier",         PD_MULT_GRID),
    ("cost_of_funds_annual",  COF_GRID),
    ("acquisition_cost",      ACQ_GRID),
    ("lgd",                   LGD_GRID),
    ("apr_strategy",          APR_STRATS),
]:
    print(f"\nVarying {p}:")
    print(vary(p, vals).to_string(index=False))

Holding other params at center (mult=2, COF=0.03, acq=250, LGD=0.65, APR=tiered_cap_24):


Varying pd_multiplier:
 pd_multiplier  k_star_approve_pct  mean_Expected_Profit    category
           1.0          100.000000            900.808042 approve_all
           2.0           99.253289            999.636277 approve_all
           3.0           97.733591           1037.032965    interior
           5.0           95.558296           1036.985057    interior

Varying cost_of_funds_annual:
 cost_of_funds_annual  k_star_approve_pct  mean_Expected_Profit    category
                 0.00           99.652495           1297.602202 approve_all
                 0.03           99.253289            999.636277 approve_all
                 0.06           98.701519            701.670351    interior

Varying acquisition_cost:
 acquisition_cost  k_star_approve_pct  mean_Expected_Profit    category
                0           99.690636           1249.636277 approve_all
              250           99.2532

## 6. Profit-vs-Youden cutoff finding under stress

In [7]:
# Distribution of cutoff_gap (k_star - youden_pct) across all 576 cells
gap = stress["cutoff_gap_profit_minus_youden"]
print("cutoff_gap = k_star_approve_pct - youden_approve_pct  (PRIMARY=LightGBM)")
print(f"  min:    {gap.min():>+7.2f} pp")
print(f"  p10:    {gap.quantile(0.10):>+7.2f} pp")
print(f"  median: {gap.median():>+7.2f} pp")
print(f"  p90:    {gap.quantile(0.90):>+7.2f} pp")
print(f"  max:    {gap.max():>+7.2f} pp")
print(f"  mean:   {gap.mean():>+7.2f} pp")
print()
print(f"  Cells where profit cutoff is MORE PERMISSIVE than Youden: {(gap > 0).sum()} / {len(gap)} = {(gap > 0).mean():.1%}")
print(f"  Cells where profit cutoff is LESS PERMISSIVE than Youden: {(gap < 0).sum()} / {len(gap)} = {(gap < 0).mean():.1%}")
print(f"  Cells where they match (within 1 pp):                     {(gap.abs() <= 1).sum()} / {len(gap)} = {(gap.abs() <= 1).mean():.1%}")

# By category
print()
print("Mean cutoff_gap by category:")
print(stress.groupby("category")["cutoff_gap_profit_minus_youden"].agg(["mean", "median", "min", "max"]).round(2).to_string())

cutoff_gap = k_star_approve_pct - youden_approve_pct  (PRIMARY=LightGBM)
  min:      +5.33 pp
  p10:     +15.00 pp
  median:  +18.83 pp
  p90:     +20.59 pp
  max:     +20.59 pp
  mean:    +18.23 pp

  Cells where profit cutoff is MORE PERMISSIVE than Youden: 576 / 576 = 100.0%
  Cells where profit cutoff is LESS PERMISSIVE than Youden: 0 / 576 = 0.0%
  Cells where they match (within 1 pp):                     0 / 576 = 0.0%

Mean cutoff_gap by category:
              mean  median    min    max
category                                
approve_all  20.41   20.58  19.63  20.59
interior     16.89   17.32   5.33  19.58


## 7. Three anchor scenarios

In [8]:
# OPTIMISTIC base: mult=1, COF=0, acq=0, LGD=0.55, tiered_uncapped
opt = stress[
    (stress["pd_multiplier"]==1.0) & (stress["cost_of_funds_annual"]==0.0) &
    (stress["acquisition_cost"]==0) & (stress["lgd"]==0.55) &
    (stress["apr_strategy"]=="tiered_uncapped")
].iloc[0]

# REALISTIC central: mult=2, COF=0.03, acq=250, LGD=0.65, tiered_cap_24
real = stress[
    (stress["pd_multiplier"]==2.0) & (stress["cost_of_funds_annual"]==0.03) &
    (stress["acquisition_cost"]==250) & (stress["lgd"]==0.65) &
    (stress["apr_strategy"]=="tiered_cap_24")
].iloc[0]

# ADVERSE stress: mult=5, COF=0.06, acq=500, LGD=0.85, flat_18
adv = stress[
    (stress["pd_multiplier"]==5.0) & (stress["cost_of_funds_annual"]==0.06) &
    (stress["acquisition_cost"]==500) & (stress["lgd"]==0.85) &
    (stress["apr_strategy"]=="flat_18")
].iloc[0]

anchors = {
    "optimistic_base": opt.to_dict(),
    "realistic_central": real.to_dict(),
    "adverse_stress": adv.to_dict(),
}
print("3 anchor scenarios:")
for name, vals in anchors.items():
    print(f"\n{name}:")
    for k, v in vals.items():
        if isinstance(v, float):
            print(f"  {k:<35} {v:>15.4f}")
        else:
            print(f"  {k:<35} {v}")

3 anchor scenarios:

optimistic_base:
  pd_multiplier                                1.0000
  apr_strategy                        tiered_uncapped
  cost_of_funds_annual                         0.0000
  acquisition_cost                    0
  lgd                                          0.5500
  mean_pd_stressed                             0.0095
  mean_LT_EL                                  53.2676
  mean_LT_margin                            1556.0200
  mean_Expected_Profit                      1502.7523
  total_Expected_Profit                354601464.4394
  share_profit_gt_0                            1.0000
  k_star_approve_pct                         100.0000
  cutoff_pd_star                               0.1916
  max_total_profit_at_k_star           354601464.4394
  youden_approve_pct_full                     79.4057
  cutoff_gap_profit_minus_youden              20.5943
  category                            approve_all

realistic_central:
  pd_multiplier                           

## 8. Save artifacts

In [9]:
OUT_DIR = REPO_ROOT / "artifacts/economic_framework"
OUT_DIR.mkdir(parents=True, exist_ok=True)

# Stress grid as parquet
stress.to_parquet(OUT_DIR / "economic_stress_grid.parquet", index=False)
print(f"Saved: {OUT_DIR / 'economic_stress_grid.parquet'}  ({len(stress)} cells)")

# Cut-off-only CSV (compact)
stress[["pd_multiplier","apr_strategy","cost_of_funds_annual","acquisition_cost","lgd",
         "k_star_approve_pct","cutoff_pd_star","max_total_profit_at_k_star",
         "youden_approve_pct_full","cutoff_gap_profit_minus_youden","category"]].to_csv(
    OUT_DIR / "stress_cutoff_results.csv", index=False)
print(f"Saved: {OUT_DIR / 'stress_cutoff_results.csv'}")

# Anchor scenarios JSON
with open(OUT_DIR / "anchor_scenarios.json", "w") as f:
    json.dump({k: {kk: (float(vv) if hasattr(vv, "item") else vv) for kk, vv in v.items()}
               for k, v in anchors.items()}, f, indent=2, default=str)
print(f"Saved: {OUT_DIR / 'anchor_scenarios.json'}")

print(f"\nTotal Phase 3.2 wall: {time.time()-T0:.1f}s")

Saved: E:\Study\rl-debt-collection\.claude\worktrees\funny-dubinsky\artifacts\economic_framework\economic_stress_grid.parquet  (576 cells)
Saved: E:\Study\rl-debt-collection\.claude\worktrees\funny-dubinsky\artifacts\economic_framework\stress_cutoff_results.csv
Saved: E:\Study\rl-debt-collection\.claude\worktrees\funny-dubinsky\artifacts\economic_framework\anchor_scenarios.json

Total Phase 3.2 wall: 135.9s
